# CIFAR-10 CNN Mini-Project (Day 6)
Progression: MNIST -> Cats vs Dogs -> **CIFAR-10** (RGB, 10 classes) -> transfer learning (optional).

Steps: load and inspect, split, build a CNN, check feature-map sizes, train, evaluate,
error analysis, data augmentation, results table, optional transfer learning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras import Input, Sequential, Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Rescaling, Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    RandomFlip, RandomRotation, RandomZoom,
    Resizing, GlobalAveragePooling2D,
)

SEED = 42
EPOCHS = 30          # upper limit; EarlyStopping usually stops sooner
BATCH_SIZE = 64
tf.keras.utils.set_random_seed(SEED)

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

## 1. Load and inspect the data
Images stay as raw `uint8` (0-255). Scaling happens *inside* the model (`Rescaling`),
so the same data also works for transfer learning later.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

print("X_train:", X_train_full.shape, X_train_full.dtype)   # (50000, 32, 32, 3) uint8
print("y_train:", y_train_full.shape)                       # (50000,)
print("X_test: ", X_test.shape)                             # (10000, 32, 32, 3)
print("y_test: ", y_test.shape)                             # (10000,)
# 32 x 32 x 3  ->  height x width x RGB channels

In [ ]:
def show_images(images, titles, rows, cols, figsize=(10, 6)):
    """Display a grid of images with titles."""
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        if i >= rows * cols:
            break
        plt.subplot(rows, cols, i + 1)
        plt.imshow(img)
        plt.title(title, fontsize=9)
        plt.axis("off")
    plt.tight_layout()
    plt.show()


show_images(
    X_train_full[:12],
    [CLASS_NAMES[i] for i in y_train_full[:12]],
    rows=3, cols=4,
)

## 2. Train / validation split
10% of the training set becomes validation data (stratified: every class keeps the same share).

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full,
    test_size=0.1, random_state=SEED, stratify=y_train_full,
)
print("train:", X_train.shape, "| val:", X_val.shape, "| test:", X_test.shape)

## 3. Model builder
One function defines the CNN. The only difference between the baseline and the
augmented model is `augment=True`.

In [ ]:
def build_model(augment=False, dropout=0.5):
    model = Sequential(name="cnn_aug" if augment else "cnn_base")
    model.add(Input(shape=(32, 32, 3)))
    if augment:  # these layers are only active during training
        model.add(RandomFlip("horizontal"))
        model.add(RandomRotation(0.1))
        model.add(RandomZoom(0.1))
    model.add(Rescaling(1.0 / 255))
    for filters in (32, 64, 128):
        model.add(Conv2D(filters, (3, 3), activation="relu"))
        model.add(MaxPooling2D((2, 2)))
    model.add(Flatten())
    model.add(Dense(128, activation="relu"))
    model.add(Dropout(dropout))
    model.add(Dense(10, activation="softmax"))
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",   # integer labels + softmax over 10 classes
        metrics=["accuracy"],
    )
    return model


model = build_model()
model.summary()

### Board exercise: feature-map sizes
Fill in by hand, then compare with `model.summary()` above.

| Layer | Output shape |
|---|---|
| Input | 32 x 32 x 3 |
| Conv2D(32, 3x3) | 30 x 30 x 32 |
| MaxPooling | 15 x 15 x 32 |
| Conv2D(64, 3x3) | 13 x 13 x 64 |
| MaxPooling | 6 x 6 x 64 |
| Conv2D(128, 3x3) | 4 x 4 x 128 |
| MaxPooling | 2 x 2 x 128 |
| Flatten | 512 |

Rule: a 3x3 "valid" convolution removes 2 pixels per dimension; 2x2 pooling halves (rounding down).

## 4. Train the baseline CNN

In [ ]:
def plot_history(history, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["accuracy", "loss"]):
        ax.plot(history.history[metric], label="train")
        ax.plot(history.history["val_" + metric], label="validation")
        ax.set_xlabel("epoch")
        ax.set_title(f"{title} {metric}".strip())
        ax.legend()
    plt.tight_layout()
    plt.show()


early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history_base = model.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=2,
)
plot_history(history_base, "baseline")

What to look for: a growing gap between training and validation accuracy means **overfitting**.

## 5. Evaluate on the test set

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Baseline test accuracy: {test_acc:.4f}")

probs = model.predict(X_test, verbose=0)
y_pred = probs.argmax(axis=1)

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3))

## 6. Confusion matrix
Rows are normalized, so each cell is the fraction of that *actual* class predicted as the column class.

In [ ]:
def plot_confusion(y_true, y_hat, title):
    cm = confusion_matrix(y_true, y_hat)
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()
    return cm


cm = plot_confusion(y_test, y_pred, "CIFAR-10 confusion matrix (baseline)")

# Most confused pairs (largest off-diagonal counts)
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)
for idx in np.dstack(np.unravel_index(np.argsort(-off_diag, axis=None), off_diag.shape))[0][:5]:
    a, p = idx
    print(f"{CLASS_NAMES[a]:>10} predicted as {CLASS_NAMES[p]:<10}: {off_diag[a, p]}")

## 7. Error analysis: look at the mistakes

In [ ]:
wrong = np.where(y_test != y_pred)[0]
print(f"{len(wrong)} wrong predictions out of {len(y_test)}")

show_images(
    X_test[wrong[:12]],
    [f"True: {CLASS_NAMES[y_test[i]]}\nPred: {CLASS_NAMES[y_pred[i]]}" for i in wrong[:12]],
    rows=3, cols=4, figsize=(12, 8),
)

In [ ]:
# Confidently wrong: the model was sure, and still wrong.
confidence = probs.max(axis=1)
confident_wrong = wrong[np.argsort(-confidence[wrong])][:12]

show_images(
    X_test[confident_wrong],
    [f"True: {CLASS_NAMES[y_test[i]]}\nPred: {CLASS_NAMES[y_pred[i]]} ({confidence[i]:.0%})"
     for i in confident_wrong],
    rows=3, cols=4, figsize=(12, 8),
)

## 8. Data augmentation
Same architecture, same seed, same epoch limit. Only `augment=True` differs.

In [ ]:
tf.keras.utils.set_random_seed(SEED)
model_aug = build_model(augment=True)
model_aug.summary()

history_aug = model_aug.fit(
    X_train, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=2,
)
plot_history(history_aug, "augmented")

In [ ]:
# Preview what augmentation does to a single image
aug_layers = Sequential([RandomFlip("horizontal"), RandomRotation(0.1), RandomZoom(0.1)])
sample = X_train[:1]
show_images(
    [sample[0]] + [aug_layers(sample, training=True)[0].numpy().astype("uint8") for _ in range(7)],
    ["original"] + [f"aug {i}" for i in range(1, 8)],
    rows=2, cols=4, figsize=(10, 5),
)

In [ ]:
aug_acc = model_aug.evaluate(X_test, y_test, verbose=0)[1]
print(f"Augmented test accuracy: {aug_acc:.4f}")

y_pred_aug = model_aug.predict(X_test, verbose=0).argmax(axis=1)
plot_confusion(y_test, y_pred_aug, "CIFAR-10 confusion matrix (augmented)")

## 9. Compare results

In [ ]:
def best_val_acc(history):
    return max(history.history["val_accuracy"])


results = pd.DataFrame([
    {"model": "CNN",
     "params": model.count_params(),
     "epochs run": len(history_base.history["loss"]),
     "best val acc": best_val_acc(history_base),
     "test acc": test_acc},
    {"model": "CNN + augmentation",
     "params": model_aug.count_params(),
     "epochs run": len(history_aug.history["loss"]),
     "best val acc": best_val_acc(history_aug),
     "test acc": aug_acc},
])
results

## 10. (Optional, advanced) Transfer learning with MobileNetV2
Pipeline: pretrained CNN -> learned features -> new classification head -> 10 CIFAR-10 classes.
CIFAR-10 images are tiny, so they are resized to 96x96 first. MobileNetV2 expects pixels in [-1, 1].
Note: `weights="imagenet"` downloads the pretrained weights (needs internet).

In [ ]:
from tensorflow.keras.applications import MobileNetV2

def build_transfer_model():
    base = MobileNetV2(input_shape=(96, 96, 3), include_top=False, weights="imagenet")
    base.trainable = False                       # freeze the pretrained features

    inputs = Input(shape=(32, 32, 3))
    x = RandomFlip("horizontal")(inputs)
    x = Resizing(96, 96)(x)
    x = Rescaling(scale=1 / 127.5, offset=-1)(x)  # [0, 255] -> [-1, 1]
    x = base(x, training=False)                  # keep BatchNorm in inference mode
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.2)(x)
    outputs = Dense(10, activation="softmax")(x)

    model = Model(inputs, outputs, name="mobilenetv2_transfer")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model, base


tf.keras.utils.set_random_seed(SEED)
tl_model, base_model = build_transfer_model()
history_tl = tl_model.fit(
    X_train, y_train,
    epochs=5, batch_size=128,
    validation_data=(X_val, y_val),
    verbose=2,
)
plot_history(history_tl, "transfer")

tl_acc = tl_model.evaluate(X_test, y_test, verbose=0)[1]
print(f"Transfer-learning test accuracy: {tl_acc:.4f}")

## 11. Fine-tuning: unfreeze the top of MobileNetV2
Stage 1 (above) trained only the new head. Stage 2 also adapts the **last layers** of the
pretrained network to CIFAR-10, using a **very small learning rate** so the useful ImageNet
features are nudged rather than destroyed.

Rules for fine-tuning:
1. Train the head first (done), so the random head does not send large gradients into the base.
2. Unfreeze only the top layers; early layers hold generic features (edges, textures).
3. **Recompile** after changing `trainable`, and use a learning rate ~100x smaller (1e-5).
4. Keep BatchNorm layers frozen so their statistics are not disturbed by small batches.

In [ ]:
UNFREEZE_LAST = 30   # number of layers at the end of MobileNetV2 to fine-tune
FT_EPOCHS = 10       # additional epochs

print("Layers in base model:", len(base_model.layers))
print("Trainable params before:", int(np.sum([np.prod(w.shape) for w in tl_model.trainable_weights])))

base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAST]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompiling is required for the change to take effect
tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
print("Trainable params after: ", int(np.sum([np.prod(w.shape) for w in tl_model.trainable_weights])))

In [ ]:
stage1_epochs = len(history_tl.epoch)

history_ft = tl_model.fit(
    X_train, y_train,
    epochs=stage1_epochs + FT_EPOCHS,
    initial_epoch=stage1_epochs,          # continue counting from where stage 1 stopped
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
    verbose=2,
)

In [ ]:
# Curves for both stages on one plot; the dashed line marks where fine-tuning began
def merge_histories(h1, h2):
    return {k: h1.history[k] + h2.history[k] for k in h1.history}

merged = merge_histories(history_tl, history_ft)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric in zip(axes, ["accuracy", "loss"]):
    ax.plot(merged[metric], label="train")
    ax.plot(merged["val_" + metric], label="validation")
    ax.axvline(stage1_epochs - 0.5, color="gray", linestyle="--", label="start fine-tuning")
    ax.set_xlabel("epoch")
    ax.set_title(metric)
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
ft_acc = tl_model.evaluate(X_test, y_test, verbose=0)[1]
print(f"Transfer learning (frozen base): {tl_acc:.4f}")
print(f"After fine-tuning:               {ft_acc:.4f}")

y_pred_ft = tl_model.predict(X_test, verbose=0).argmax(axis=1)
plot_confusion(y_test, y_pred_ft, "CIFAR-10 confusion matrix (fine-tuned MobileNetV2)")

In [ ]:
# Final comparison of all models
final_results = pd.concat([
    results,
    pd.DataFrame([
        {"model": "MobileNetV2 (frozen base)", "params": None,
         "epochs run": stage1_epochs,
         "best val acc": max(history_tl.history["val_accuracy"]), "test acc": tl_acc},
        {"model": "MobileNetV2 (fine-tuned)", "params": tl_model.count_params(),
         "epochs run": stage1_epochs + len(history_ft.epoch),
         "best val acc": max(merged["val_accuracy"]), "test acc": ft_acc},
    ]),
], ignore_index=True)
final_results

Note: the frozen-base row has no separate parameter count because it is the same network as the
fine-tuned row (only which weights were trainable changed).

## Discussion questions
1. Which classes does the CNN confuse most, and why might they look alike at 32x32?
2. Did augmentation reduce the train/validation gap? Did it improve test accuracy?
3. Why does the pretrained model need far fewer epochs?
4. Why do we use a much smaller learning rate (1e-5) for fine-tuning than for the head (1e-3)?
5. Did fine-tuning help? What could go wrong if you unfroze *all* layers immediately?